In [2]:
import os
from typing import List, Dict, Optional

from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import JsonOutputParser
# from langchain_core.pydantic_v1 import BaseModel, Field
from pydantic import BaseModel, Field

def run_analysis():
    """
    Main execution function for the advanced GPR risk analysis pipeline.
    """
    # --- Step 1: Load Environment Variables ---
    # This line loads the OPENAI_API_KEY from the .env file into the environment.
    # The ChatOpenAI class will automatically find and use this key.
    load_dotenv()
    if not os.getenv("OPENAI_API_KEY"):
        print("Error: OPENAI_API_KEY is not set. Please set it in your .env file or as an environment variable.")
        return

    # --- Step 2: Define GPR Risk Categories (in English) ---
    # GPR_CATEGORIES: Dict[str, str] = {
    #     "War Threats": "Mentions of potential armed conflicts or significant interstate tensions.",
    #     "Peace Threats": "Mentions of threats to peace agreements or regional stability.",
    #     "Military Buildups": "Discussions of armament, troop movements, or increases in military capacity.",
    #     "Nuclear Threats": "Explicit mentions of nuclear weapons or capabilities as a threat.",
    #     "Terror Threats": "Mentions of potential terrorist activities or threats from terrorist groups.",
    #     "Beginning of War": "Reports on the outbreak of a new war or conflict.",
    #     "Escalation of War": "Reports on the intensification of an existing conflict.",
    #     "Terror Acts": "Reports of specific terrorist attacks that have occurred."
    # }

    # RISK_CATEGORIES: Dict[str, str] = {
    #     "俄乌冲突": "俄乌冲突的描述（具体发生时间）", “受影响的行业”，
    #     "COVID_19": "COVID的描述 （大概时间）", “受影响的行业”，
    #     "911": "911的描述 （具体时间）", “受影响的行业”， # Control 例子 （看是否影响，对照组）
    # }

    RISK_CATEGORIES: Dict[str, str] = {
        "Russia_Ukraine_Conflict": "Mentions of the Russia-Ukraine conflict, including its timeline, specific events, and its impact on industries such as supply chains, energy, or finance.",
        "COVID_19_Pandemic": "Mentions of the COVID-19 pandemic, its general timeline (e.g., outbreaks, lockdowns), and its effects on industries like healthcare, travel, labor, or manufacturing.",
        "September_11_Attacks": "Mentions of the September 11th, 2001 attacks, their specific timing, and any described long-term or comparative impacts on industries, particularly aviation, security, and insurance. This can act as a control case for historical risk discussion.",
    }

    # --- Step 3: Define the Desired, Structured Output Format (Pydantic Models) ---
    class RiskAnalysis(BaseModel):
        """Analysis for a single risk category, with explicit calculation fields."""
        category: str = Field(description="The GPR risk category being analyzed.")
        baseline_score: int = Field(description="The initial baseline score selected (0, 1, 21, 51, or 81).")
        adjustment_points: int = Field(description="Points added or subtracted from the baseline. Can be positive or negative.")
        final_score: int = Field(description="The final calculated score (baseline + adjustment).", ge=0, le=100)
        reasoning: str = Field(description="The detailed causal argument, which MUST explain the choice of baseline and the reason for the adjustment points.")

    class HighestRiskEvidence(BaseModel):
        """Evidence for the highest-scored risk category."""
        category: str = Field(description="The name of the category with the highest risk score.")
        relevant_sentences: List[str] = Field(description="A list of the most relevant original sentences from the source text that support the high score.")

    class GPRFinalReport(BaseModel):
        """The final, complete GPR risk report with explicit calculations."""
        risk_analysis: List[RiskAnalysis] = Field(description="A list of analyses for all 8 GPR categories.")
        highest_risk_evidence: HighestRiskEvidence = Field(description="Evidence for the single highest-scored risk.")

    # --- Step 4: Initialize the Model and Output Parser ---
    try:
        llm = ChatOpenAI(model="gpt-4o", temperature=0.0, model_kwargs={"response_format": {"type": "json_object"}})
        parser = JsonOutputParser(pydantic_object=GPRFinalReport)
    except Exception as e:
        print(f"An error occurred during model or parser initialization: {e}")
        return

    # --- Step 5: Create the Core Chain of Thought Prompt Template with Corrected Braces ---
    prompt_template = """
    You are a hyper-precise geopolitical risk analyst. Your task is to conduct a risk assessment based on the provided "Item 1a - Risk Factors" text. You MUST follow all rules meticulously.

    **GOLDEN RULE #1: THE ZERO SCORE**
    If, after scanning the text, a risk category is COMPLETELY ABSENT (no direct, indirect, or boilerplate mention), the score MUST be exactly 0. In this case, `baseline_score`, `adjustment_points`, and `final_score` must all be 0. Do NOT assign a score of 1. THIS IS YOUR MOST IMPORTANT RULE.

    **SCORING METHODOLOGY: "Baseline and Adjust"**
    For any risk that IS mentioned, you must follow this two-step calculation:

    1.  **Select a Baseline:** Choose a `baseline_score` from the list below. This is the STARTING POINT.
        * **Baseline 1:** For vague, boilerplate mentions of general global risks.
        * **Baseline 21:** For direct mentions of a risk, but with low, hypothetical, or non-material impact.
        * **Baseline 51:** For significant discussion of a direct risk with potential material impact.
        * **Baseline 81:** For risks presented as a central, severe, and immediate threat.

    2.  **Calculate Adjustment:** Based on textual evidence, determine the `adjustment_points`.
        * **Add points** for aggravating factors (e.g., specific examples, repeated mentions, quantified financial impact).
        * **Subtract points** for mitigating factors (e.g., strong countermeasures described, risk is remote).
        * The final score MUST stay within the tier's logical range (e.g., a baseline 21 risk should end up between 21-50).

    3.  **Calculate Final Score:** `final_score` = `baseline_score` + `adjustment_points`.

    **FEW-SHOT EXAMPLES (How your brain should work):**

    * **Example for a non-zero, adjusted score:**
        * *Thought Process:* "The text mentions 'conflict in Eastern Europe' impacting supply chains (Baseline 51). It also quantifies a potential $10M loss, which is a strong aggravating factor. I will add 15 points."
        * *JSON Output Snippet for this risk:*
            ```json
            {{
                "category": "War Threats",
                "baseline_score": 51,
                "adjustment_points": 15,
                "final_score": 66,
                "reasoning": "Baseline set to 51 due to direct mention of conflict impacting the company. Adjustment of +15 points is added because the text quantifies a specific financial impact of $10M, indicating a high level of materiality."
            }}
            ```

    * **Example for a zero score:**
        * *Thought Process:* "I have scanned the entire document for keywords like 'nuclear', 'atomic', 'warhead'. There are zero mentions. This risk is absent."
        * *JSON Output Snippet for this risk:*
            ```json
            {{
                "category": "Nuclear Threats",
                "baseline_score": 0,
                "adjustment_points": 0,
                "final_score": 0,
                "reasoning": "The document contains no direct or indirect mentions of nuclear threats. Per the Golden Rule, the score is 0."
            }}
            ```

    **YOUR TASK:**
    Now, apply this exact methodology to the `DOCUMENT TEXT` below for all 8 GPR categories. Produce a single, valid JSON object according to the required format.

    **GPR CATEGORIES TO ASSESS:**
    {categories}

    **DOCUMENT TEXT (Item 1a):**
    ```text
    {document_text}
    ```

    **REQUIRED JSON OUTPUT FORMAT:**
    {format_instructions}
    """

    # 计算

    categories_text = "\n".join([f"- **{name}**: {desc}" for name, desc in RISK_CATEGORIES.items()])
    prompt = PromptTemplate(
        template=prompt_template,
        input_variables=["document_text"],
        partial_variables={
            "format_instructions": parser.get_format_instructions(),
            "categories": categories_text
        }
    )

    # --- Step 6: Construct and Run the Langchain Chain ---
    chain = prompt | llm | parser

    # Load the text content from the Item 1a file.
    try:
        with open("item1a_text.txt", "r", encoding="utf-8") as f:
            item1a_content = f.read()
    except FileNotFoundError:
        print("Error: `item1a_text.txt` not found. Please create this file and paste your Item 1a text into it.")
        return

    print("--- Running Advanced GPR Risk Analysis with GPT-4o... This may take a moment. ---")
    try:
        report_data = chain.invoke({"document_text": item1a_content})
        print("--- Analysis Complete. Formatting Report... ---\n")
    except Exception as e:
        print(f"An error occurred while invoking the Langchain chain: {e}")
        return
        
    # --- Step 7: Format and Print the Final Report ---
    print_report(report_data)

def print_report(report_data: Optional[Dict]):
    """
    A helper function to print the V2 analysis report in a readable format.
    """
    if not report_data:
        print("Failed to generate the report.")
        return

    print("==============================================")
    print("      GEOPOLITICAL RISK (GPR) REPORT (V2)      ")
    print("==============================================")
    
    # Sort the risk analysis results by score in descending order for better readability.
    sorted_analysis = sorted(report_data.get('risk_analysis', []), key=lambda x: x.get('final_score', 0), reverse=True)

    for analysis in sorted_analysis:
        cat = analysis.get('category', 'N/A')
        final = analysis.get('final_score', 'N/A')
        base = analysis.get('baseline_score', 'N/A')
        adj = analysis.get('adjustment_points', 'N/A')
        reason = analysis.get('reasoning', 'N/A')
        
        print(f"\n--- Risk Category: {cat} ---")
        print(f"  Final Score (0-100): {final}")
        print(f"  Calculation: {base} (Baseline) + {adj} (Adjustment)")
        print(f"  Causal Reasoning: {reason}")
    
    print("\n\n==============================================")
    print("      HIGHEST RISK IDENTIFIED      ")
    print("==============================================")
    
    highest_risk = report_data.get('highest_risk_evidence')
    if highest_risk:
        print(f"\nCategory with Highest Score: {highest_risk.get('category', 'N/A')}")
        print("\nMost Relevant Original Sentences from Document:")
        sentences = highest_risk.get('relevant_sentences', [])
        if sentences:
            for i, sentence in enumerate(sentences, 1):
                print(f'  {i}. "{sentence}"')
        else:
            print("  (No relevant sentences were extracted.)")
    else:
        print("  (Could not identify the highest risk item.)")
        
    print("\n==============================================")


if __name__ == "__main__":
    run_analysis()


# 评分从0-1 到线性评分。
# 输出所有相关句。
# 对相关句子的 

--- Running Advanced GPR Risk Analysis with GPT-4o... This may take a moment. ---
--- Analysis Complete. Formatting Report... ---

      GEOPOLITICAL RISK (GPR) REPORT (V2)      

--- Risk Category: COVID_19_Pandemic ---
  Final Score (0-100): 61
  Calculation: 51 (Baseline) + 10 (Adjustment)
  Causal Reasoning: Baseline set to 51 due to direct mention of COVID-19 affecting advertising revenues and financial results. Adjustment of +10 points is added because the text discusses the pandemic's impact on revenue growth and market volatility, indicating a significant material impact.

--- Risk Category: Russia_Ukraine_Conflict ---
  Final Score (0-100): 0
  Calculation: 0 (Baseline) + 0 (Adjustment)
  Causal Reasoning: The document contains no direct or indirect mentions of the Russia-Ukraine conflict. Per the Golden Rule, the score is 0.

--- Risk Category: September_11_Attacks ---
  Final Score (0-100): 0
  Calculation: 0 (Baseline) + 0 (Adjustment)
  Causal Reasoning: The document conta

In [6]:
RISK_CATEGORIES: Dict[str, str] = {
    "Russia_Ukraine_Conflict": "Mentions of the Russia-Ukraine conflict, including its timeline, specific events, and its impact on industries such as supply chains, energy, or finance.",
    "COVID_19_Pandemic": "Mentions of the COVID-19 pandemic, its general timeline (e.g., outbreaks, lockdowns), and its effects on industries like healthcare, travel, labor, or manufacturing.",
    "September_11_Attacks": "Mentions of the September 11th, 2001 attacks, their specific timing, and any described long-term or comparative impacts on industries, particularly aviation, security, and insurance. This can act as a control case for historical risk discussion.",
}

# --- Step 3: Define the Desired, Structured Output Format (Pydantic Models) ---
class RiskAnalysis(BaseModel):
    """Analysis for a single risk category, with explicit calculation fields."""
    category: str = Field(description="The GPR risk category being analyzed.")
    baseline_score: int = Field(description="The initial baseline score selected (0, 1, 21, 51, or 81).")
    adjustment_points: int = Field(description="Points added or subtracted from the baseline. Can be positive or negative.")
    final_score: int = Field(description="The final calculated score (baseline + adjustment).", ge=0, le=100)
    reasoning: str = Field(description="The detailed causal argument, which MUST explain the choice of baseline and the reason for the adjustment points.")

class HighestRiskEvidence(BaseModel):
    """Evidence for the highest-scored risk category."""
    category: str = Field(description="The name of the category with the highest risk score.")
    relevant_sentences: List[str] = Field(description="A list of the most relevant original sentences from the source text that support the high score.")

class GPRFinalReport(BaseModel):
    """The final, complete GPR risk report with explicit calculations."""
    risk_analysis: List[RiskAnalysis] = Field(description="A list of analyses for all 8 GPR categories.")
    highest_risk_evidence: HighestRiskEvidence = Field(description="Evidence for the single highest-scored risk.")

# --- Step 4: Initialize the Model and Output Parser ---

# llm = ChatOpenAI(model="gpt-4o", temperature=0.0, model_kwargs={"response_format": {"type": "json_object"}})
parser = JsonOutputParser(pydantic_object=GPRFinalReport)
with open('test.txt', 'w') as f:
    f.write(parser.get_format_instructions())
# parser.get_format_instructions()